# Fine-tuning `SARFloodUNet` for SHELTERProduces `sar_flood.pt` — the state dict that moves the flood stage from thresholdheuristics (confidence `0.55`) to a trained model (confidence `0.88`).**Runs unchanged on:** your Mac (Apple Silicon / MPS), Google Colab (CUDA), Kaggle(CUDA), or any CPU box. The device is auto-detected in cell 1.---## What this notebook does1. Downloads **Sen1Floods11** — 446 hand-labelled Sentinel-1 chips, CC-BY, no account2. Rebuilds the *exact* model and preprocessing from the SHELTER backend3. Holds out **Nigeria entirely** as the test set — the honest generalisation question4. Trains with a masked BCE + Dice loss5. **Benchmarks against the −16 dB heuristic it would replace**6. Exports `sar_flood.pt` and verifies it loads the way the backend loads it## The one rule that mattersThe backend will load these weights and run them through *its* preprocessing, not thisnotebook's. If the two disagree by even a scale factor, **the weights load withouterror and predict nonsense** — the worst failure mode available, because`confidence` would report `0.88` while the output is garbage.So `SARFloodUNet` and `_standardize` below are copied **verbatim** from`backend/app/ml/models.py` and `backend/app/ml/inference.py`. Cell 3 checks themagainst the live backend when you run this inside the repo. **Do not "improve" them.**## Ship/no-ship gateSection 8 asserts a trained model must beat the physical threshold *on held-outNigerian scenes* before it ships. Cell 11 computes exactly that. If it fails, thecorrect action is to keep the heuristic — the backend already degrades to it cleanly,and a worse model reporting higher confidence is strictly harmful.

---## 0 · Running this on your MacBook (recommended) — or Colab / Kaggle### Option A — locally on Apple Silicon (M-series)**This is the faster path for you**, and it avoids Colab's session timeouts andre-downloading the dataset each time. PyTorch uses the integrated GPU via **MPS**.```bashcd backend/notebookspython3 -m venv .venv && source .venv/bin/activatepip install torch torchvision numpy rasterio matplotlib jupyterlabjupyter lab finetune_sar_flood_unet.ipynb```Then **Run All**. The dataset (~400 MB) caches in `sen1floods11/` beside the notebook,so re-runs skip the download.Verified on an **M4 Pro**: MPS available, ~**0.23 s** per 512² training step.Two MPS caveats already handled in the cells below — noted so they don't surprise you:- `num_workers=0` in the DataLoader. Fork-based workers plus Metal is a known hang.- `PYTORCH_ENABLE_MPS_FALLBACK=1`, so any op without an MPS kernel silently uses CPU  instead of raising.**Do not install `torch` into the backend's own venv for this.** Keep it in`notebooks/.venv` — the training dependencies (`matplotlib`, `jupyterlab`) have nobusiness in the API image.### Option B — Google ColabUpload the notebook, then **Runtime → Change runtime type → T4 GPU** (free tier) andRun All. Cell 1 installs `rasterio`. The dataset re-downloads each session, so keep thetab alive. Download the artefact at the end with:```pythonfrom google.colab import files; files.download("sar_flood.pt")```### Option C — KaggleNew Notebook → upload → **Settings → Accelerator → GPU T4 ×2**. 30 GPU-hours/week free.`sar_flood.pt` appears in the **Output** tab.> **On Colab/Kaggle the contract check in cell 2b is skipped** — it needs the backend> source, which isn't there. That is expected. The definitions were copied verbatim; the> only rule is not to edit them.### Cost either wayZero. Sen1Floods11 is CC-BY from a public bucket, both free tiers suffice, and locallyit is just your own machine for under an hour.

## 1 · Environment and device

In [ ]:
import subprocess, sys, importlib# rasterio reads the GeoTIFF chips; the rest ships with Colab/Kaggle.for pkg, pip_name in [("rasterio", "rasterio"), ("torch", "torch"),                      ("numpy", "numpy"), ("requests", "requests")]:    try:        importlib.import_module(pkg)    except ImportError:        print(f"installing {pip_name} ...")        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)import torch, numpy as np, rasterio, requestsprint("torch", torch.__version__, "| numpy", np.__version__)

In [ ]:
def pick_device() -> torch.device:    """CUDA (Colab/Kaggle) -> MPS (Apple Silicon) -> CPU. Same notebook everywhere."""    if torch.cuda.is_available():        print("CUDA:", torch.cuda.get_device_name(0))        return torch.device("cuda")    if torch.backends.mps.is_available():        print("Apple Silicon GPU via MPS")        return torch.device("mps")    print("CPU only — training will work but take hours")    return torch.device("cpu")DEVICE = pick_device()# MPS note: as of torch 2.x a few ops silently fall back to CPU. This makes any# such fallback visible instead of quietly slow. Harmless on CUDA/CPU.import osos.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")# Reproducibility. Same seed + same device => same weights.SEED = 1337torch.manual_seed(SEED); np.random.seed(SEED)if DEVICE.type == "cuda":    torch.cuda.manual_seed_all(SEED)

## 2 · Model definition — copied verbatim from the backendSource: `backend/app/ml/models.py`. If you change anything here, the exported weightswill not match what `app/ml/inference.py` instantiates.

In [ ]:
import torch.nn as nnimport torch.nn.functional as Fdef _conv_block(in_ch: int, out_ch: int) -> nn.Sequential:    """Conv-BN-ReLU twice. BatchNorm matters here because Sentinel-1 backscatter    varies by tens of dB between scenes and orbits."""    return nn.Sequential(        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),        nn.BatchNorm2d(out_ch),        nn.ReLU(inplace=True),        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),        nn.BatchNorm2d(out_ch),        nn.ReLU(inplace=True),    )class SARFloodUNet(nn.Module):    """Compact U-Net for flood segmentation from Sentinel-1 VV/VH.    Input : (B, 2, H, W) — VV and VH in decibels, standardised.    Output: (B, 1, H, W) — water logits. Apply sigmoid for probability.    """    def __init__(self, in_channels: int = 2, base: int = 16) -> None:        super().__init__()        self.enc1 = _conv_block(in_channels, base)        self.enc2 = _conv_block(base, base * 2)        self.enc3 = _conv_block(base * 2, base * 4)        self.bottleneck = _conv_block(base * 4, base * 8)        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)        self.dec3 = _conv_block(base * 8, base * 4)        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)        self.dec2 = _conv_block(base * 4, base * 2)        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)        self.dec1 = _conv_block(base * 2, base)        self.head = nn.Conv2d(base, 1, kernel_size=1)        self.pool = nn.MaxPool2d(2)    @staticmethod    def _pad_to_match(x: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:        """Odd input dimensions leave the upsampled tensor a pixel short."""        dh = ref.shape[-2] - x.shape[-2]        dw = ref.shape[-1] - x.shape[-1]        if dh or dw:            x = F.pad(x, [dw // 2, dw - dw // 2, dh // 2, dh - dh // 2])        return x    def forward(self, x: torch.Tensor) -> torch.Tensor:        e1 = self.enc1(x)        e2 = self.enc2(self.pool(e1))        e3 = self.enc3(self.pool(e2))        b = self.bottleneck(self.pool(e3))        d3 = self.dec3(torch.cat([self._pad_to_match(self.up3(b), e3), e3], dim=1))        d2 = self.dec2(torch.cat([self._pad_to_match(self.up2(d3), e2), e2], dim=1))        d1 = self.dec1(torch.cat([self._pad_to_match(self.up1(d2), e1), e1], dim=1))        return self.head(d1)# --- preprocessing, verbatim from app/ml/inference.py -----------------------# FIXED standardisation constants, shared with app/ml/inference.py.## ## Why these are constants, and the bug that proved it## This used to compute each SCENE's own mean and standard deviation. That looks reasonable and is# catastrophic for a segmentation model: water is dark in ABSOLUTE dB, and per-scene recentring maps# a bone-dry scene and a fully flooded one onto the same distribution.## Measured after the first real training run. Over Kano — Sahel, VV median -4.3 dB, the -16 dB# heuristic reporting 0.000 water — the trained model said 52.9% standing water and escalated to the# platform's first-ever WARNING at confidence 0.88. Probing the weights with fixed inputs showed the# model was fine and monotonic:##     VV  -4 dB -> P(water) 0.022        VV -20 dB -> P(water) 0.778#     VV  -8 dB -> P(water) 0.034        VV -25 dB -> P(water) 0.913## The model was right; the input was a lie. After the fix, Kano reads 0.003.## So both sides now use these constants and the contract cell below asserts they match. Divergence# here is training/serving skew — the failure that makes a good validation score meaningless.SAR_DB_MEAN = (-12.0, -19.0)   # VV, VHSAR_DB_STD = (5.0, 5.0)def _standardize(array: np.ndarray, channel: int = 0) -> np.ndarray:    """Standardise Sentinel-1 dB with FIXED constants. NaN filled post-standardisation with 0.    `channel` selects VV (0) or VH (1) — they differ by ~7 dB, so using one channel's constants for    the other shifts it wholesale.    NaN becomes 0.0, which IS the standardised mean, so a no-data pixel reads as typical rather than    as an extreme. It is excluded from every reported fraction anyway, but it still passes through    the convolution and must not drag its neighbours.    """    mean = SAR_DB_MEAN[channel]    std = SAR_DB_STD[channel] or 1.0    out = (array - mean) / std    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")def sar_water_mask(vv_db: np.ndarray, threshold_db: float = -16.0) -> np.ndarray:    """The heuristic this model must beat. Verbatim from app/eo/indices.py."""    return np.where(np.isnan(vv_db), np.nan,                    (vv_db < threshold_db).astype("float32")).astype("float32")model = SARFloodUNet()n_params = sum(p.numel() for p in model.parameters())print(f"SARFloodUNet: {n_params:,} parameters  (~{n_params*4/1e6:.1f} MB as float32)")

import ast, pathlib, textwrapdef _strip_docstrings(node):    for sub in ast.walk(node):        if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Module)):            if (sub.body and isinstance(sub.body[0], ast.Expr)                    and isinstance(sub.body[0].value, ast.Constant)                    and isinstance(sub.body[0].value.value, str)):                sub.body.pop(0)    return nodedef _defn(src: str, name: str) -> str | None:    """Structural fingerprint of a top-level class/function, docstrings removed."""    for node in ast.parse(textwrap.dedent(src)).body:        if isinstance(node, (ast.ClassDef, ast.FunctionDef)) and node.name == name:            return ast.dump(_strip_docstrings(node))    return None# Locate the backend regardless of where the notebook was opened from.backend = next((pathlib.Path(c) for c in                ("../app/ml", "backend/app/ml", "../backend/app/ml", "app/ml",                 "/kaggle/input/shelter/app/ml")                if pathlib.Path(c, "models.py").exists()), None)if backend is None:    print("Backend source not present (normal on Colab/Kaggle) — contract check skipped.")    print("The definitions above were copied verbatim; do not edit them.")else:    # This notebook's own source for the definitions, read from the .ipynb itself.    import json as _json, glob    nb_path = next(iter(glob.glob("**/finetune_sar_flood_unet.ipynb", recursive=True)                        + glob.glob("finetune_sar_flood_unet.ipynb")), None)    this_nb = ""    if nb_path:        this_nb = "\n".join("\n".join(c["source"])                            for c in _json.load(open(nb_path))["cells"]                            if c["cell_type"] == "code")    targets = [("SARFloodUNet", "models.py"), ("_conv_block", "models.py"),               ("_standardize", "inference.py")]    ok = True    for name, filename in targets:        theirs = _defn(pathlib.Path(backend, filename).read_text(), name)        mine = _defn(this_nb, name) if this_nb else None        if mine is None:            print(f"  SKIP     {name} (could not read this notebook's source)")            continue        same = theirs == mine        ok &= same        print(f"  {'MATCH   ' if same else 'MISMATCH'} {name}  ({filename})")    if not ok:        raise SystemExit(            "\nThis notebook has drifted from the backend. Weights trained now would "            "load WITHOUT error and predict nonsense, while reporting confidence 0.88. "            "Re-copy the definitions from app/ml/ before training."        )    print("\nContract OK — the notebook matches the serving path exactly.")

In [ ]:
import ast, pathlib, textwrap, json as _json, globdef _fingerprint(src: str, name: str) -> str | None:    """Structural fingerprint of a top-level class/function, docstrings stripped.    Compares structure, not text — so a reworded comment or docstring will not trip    the check, but any change to a layer, kernel size or arithmetic will.    """    for node in ast.parse(textwrap.dedent(src)).body:        if isinstance(node, (ast.ClassDef, ast.FunctionDef)) and node.name == name:            for sub in ast.walk(node):                if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):                    if (sub.body and isinstance(sub.body[0], ast.Expr)                            and isinstance(sub.body[0].value, ast.Constant)                            and isinstance(sub.body[0].value.value, str)):                        sub.body.pop(0)            return ast.dump(node)    return Nonebackend = next((pathlib.Path(c) for c in                ("../app/ml", "backend/app/ml", "../backend/app/ml", "app/ml")                if pathlib.Path(c, "models.py").exists()), None)nb_file = next(iter(glob.glob("finetune_sar_flood_unet.ipynb")                    + glob.glob("**/finetune_sar_flood_unet.ipynb", recursive=True)), None)if backend is None:    print("Backend source not present (normal on Colab/Kaggle) — contract check skipped.")    print("The definitions in the previous cell were copied verbatim; do not edit them.")elif nb_file is None:    print("Could not locate this .ipynb to read its own source — check skipped.")else:    mine = "\n".join("\n".join(c["source"])                     for c in _json.load(open(nb_file))["cells"]                     if c["cell_type"] == "code")    ok = True    for name, filename in [("SARFloodUNet", "models.py"),                           ("_conv_block", "models.py"),                           ("_standardize", "inference.py")]:        theirs = _fingerprint(pathlib.Path(backend, filename).read_text(), name)        ours = _fingerprint(mine, name)        same = theirs is not None and theirs == ours        ok &= same        print(f"  {'MATCH   ' if same else 'MISMATCH'} {name}  (app/ml/{filename})")    if not ok:        raise SystemExit(            "\nThis notebook has drifted from the backend. Weights trained now would "            "load WITHOUT error and predict nonsense, while confidence reported 0.88. "            "Re-copy the definitions from app/ml/ before training."        )    print("\nContract OK — the notebook matches the serving path exactly.")

## 3 · Dataset — Sen1Floods11446 hand-labelled 512×512 Sentinel-1 chips across 11 flood events. CC-BY-4.0, hostedin a **public** Google Cloud Storage bucket — no account, no `gcloud auth`.Two properties of this dataset that will bite you if unnoticed:- **The S1 chips are already in decibels**, 2 bands (VV, VH) — the same units  `app/eo/indices.to_db()` produces. No conversion needed, and applying one would  break the match. *Verified on `Nigeria_1095404`: float32, VV range −39.5…+10.9 dB.*- **No-data is encoded as NaN**, and the GeoTIFF declares `nodata=nan` — so  `_standardize` already excludes it by construction. The `< -900` guard below is  belt-and-braces for chips that use a sentinel instead; it is a no-op on the ones  checked.- **Labels are `{-1, 0, 1}`**, where **`-1` means *no data*, not *not water***.  Training on `-1` as a negative teaches the model that missing data is dry land.  Every loss below masks it out. *Verified on `Nigeria_1095404`: int16, values exactly  `[-1 0 1]` at 3.8% / 78.7% / 17.6% — so water really is a ~1-in-6 minority class.*We download over plain HTTPS via the GCS JSON API, so this works identically on a Macand in Colab without `gsutil`.

In [ ]:
import concurrent.futures, pathlib, timeBUCKET = "sen1floods11"PREFIX = "v1.1/data/flood_events/HandLabeled"ROOT = pathlib.Path("sen1floods11")(ROOT / "S1Hand").mkdir(parents=True, exist_ok=True)(ROOT / "LabelHand").mkdir(parents=True, exist_ok=True)def list_bucket(prefix: str) -> list[str]:    """Page the public JSON API. No credentials needed for a public bucket."""    names, token = [], None    while True:        params = {"prefix": prefix, "maxResults": 1000}        if token:            params["pageToken"] = token        r = requests.get(f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o",                         params=params, timeout=60)        r.raise_for_status()        payload = r.json()        names += [item["name"] for item in payload.get("items", [])]        token = payload.get("nextPageToken")        if not token:            return namesdef fetch(name: str) -> tuple[str, bool]:    dest = ROOT / name.split(f"{PREFIX}/")[1]    if dest.exists() and dest.stat().st_size > 0:        return name, True    url = f"https://storage.googleapis.com/{BUCKET}/{name}"    try:        r = requests.get(url, timeout=180)        r.raise_for_status()        dest.write_bytes(r.content)        return name, True    except Exception as exc:        print(f"  failed {dest.name}: {exc}")        return name, Falseprint("listing bucket ...")s1 = sorted(n for n in list_bucket(f"{PREFIX}/S1Hand/") if n.endswith(".tif"))lb = sorted(n for n in list_bucket(f"{PREFIX}/LabelHand/") if n.endswith(".tif"))print(f"  {len(s1)} S1 chips, {len(lb)} labels")t0 = time.perf_counter()with concurrent.futures.ThreadPoolExecutor(max_workers=16) as pool:    results = list(pool.map(fetch, s1 + lb))print(f"downloaded {sum(ok for _, ok in results)}/{len(results)} files "      f"in {time.perf_counter()-t0:.0f}s")

In [ ]:
# Pair chips with labels and group by flood event. Filenames look like# "Nigeria_129761_S1Hand.tif" / "Nigeria_129761_LabelHand.tif".import collections, repairs = []for chip in sorted((ROOT / "S1Hand").glob("*_S1Hand.tif")):    stem = chip.name.replace("_S1Hand.tif", "")    label = ROOT / "LabelHand" / f"{stem}_LabelHand.tif"    if label.exists():        region = re.match(r"([A-Za-z-]+)_", stem).group(1)        pairs.append({"region": region, "chip": chip, "label": label, "id": stem})by_region = collections.Counter(p["region"] for p in pairs)print(f"{len(pairs)} usable pairs across {len(by_region)} regions:\n")for region, count in by_region.most_common():    flag = "   <-- our deployment geography" if region.lower() == "nigeria" else ""    print(f"  {region:12} {count:3}{flag}")assert pairs, "no pairs found — re-run the download cell"

### 3b · Splitting by region, not at randomAdjacent chips from one flood event overlap in terrain, orbit geometry and weather.A random split therefore puts near-duplicates in both train and validation, and theresulting score is **optimistic by a wide margin** — the classic spatial-leakagetrap in EO machine learning.So the split is **by region**, and specifically:| Split | Regions | Purpose ||---|---|---|| **Test** | **Nigeria** — untouched until the final cell | The ship/no-ship decision. Our actual deployment geography || Validation | 2 held-out regions | Early stopping and threshold selection || Train | Everything else | Fitting |Nigeria never contributes a gradient and never selects a hyper-parameter. That makesthe final number an honest estimate of what a Kebbi farmer would get.

In [ ]:
TEST_REGIONS = {"nigeria"}VAL_REGIONS = {"bolivia", "somalia"}   # geographically and climatically distinctdef bucket_of(region: str) -> str:    r = region.lower()    if r in TEST_REGIONS: return "test"    if r in VAL_REGIONS:  return "val"    return "train"splits = collections.defaultdict(list)for p in pairs:    splits[bucket_of(p["region"])].append(p)for name in ("train", "val", "test"):    regions = sorted({p["region"] for p in splits[name]})    print(f"{name:5} {len(splits[name]):3} chips  {regions}")if not splits["test"]:    print("\nWARNING: no Nigeria chips found. The ship/no-ship gate will fall back to "          "the pooled validation set, which is a weaker claim. Check the region names above.")

## 4 · Dataset class — preprocessing identical to serving

In [ ]:
from torch.utils.data import Dataset, DataLoaderNO_DATA = -1          # Sen1Floods11 label for "no observation"CHIP = 512class Sen1Floods11(Dataset):    """Yields (2,H,W) standardised VV/VH, (1,H,W) label, (1,H,W) valid mask.    Preprocessing is `_standardize` per channel per chip — exactly what    `inference.predict_flood` does at serving time. Per-chip rather than a global    dataset statistic on purpose: the backend standardises one scene at a time and    has no dataset-wide mean to consult, so training on global stats would create a    train/serve skew that is invisible until deployment.    """    def __init__(self, items: list[dict], augment: bool = False):        self.items = items        self.augment = augment    def __len__(self) -> int:        return len(self.items)    def __getitem__(self, i: int):        item = self.items[i]        with rasterio.open(item["chip"]) as src:            arr = src.read().astype("float32")       # (2, H, W) — already dB        with rasterio.open(item["label"]) as src:            lab = src.read(1).astype("float32")      # (H, W) in {-1, 0, 1}        # Observed encoding is NaN (nodata=nan on the GeoTIFF), which `_standardize`        # already excludes. This guard also catches a large-negative sentinel, which        # some redistributions of the dataset use. No-op on the canonical files.        arr = np.where(arr < -900, np.nan, arr)        vv, vh = arr[0], arr[1]        stack = np.stack([_standardize(vv, 0), _standardize(vh, 1)], axis=0)        valid = (lab != NO_DATA).astype("float32")        target = np.where(lab == 1, 1.0, 0.0).astype("float32")        if self.augment:            # Flips and 90-degree rotations only. No brightness/contrast jitter:            # backscatter in dB is a calibrated physical quantity, and perturbing it            # would teach the model to ignore the very cue it needs.            k = np.random.randint(4)            if k:                stack  = np.rot90(stack,  k, axes=(1, 2)).copy()                target = np.rot90(target, k).copy()                valid  = np.rot90(valid,  k).copy()            if np.random.rand() < 0.5:                stack, target, valid = stack[:, :, ::-1].copy(), target[:, ::-1].copy(), valid[:, ::-1].copy()            if np.random.rand() < 0.5:                stack, target, valid = stack[:, ::-1].copy(), target[::-1].copy(), valid[::-1].copy()        return (torch.from_numpy(stack),                torch.from_numpy(target).unsqueeze(0),                torch.from_numpy(valid).unsqueeze(0),                np.float32(np.nanmean(np.where(np.isfinite(vv), vv, np.nan))))# num_workers=0 on MPS: fork-based workers plus Metal is a known source of hangs.NW = 0 if DEVICE.type == "mps" else 2BATCH = 8 if DEVICE.type != "cpu" else 2loaders = {    "train": DataLoader(Sen1Floods11(splits["train"], augment=True),                        batch_size=BATCH, shuffle=True, num_workers=NW, drop_last=True),    "val":   DataLoader(Sen1Floods11(splits["val"]),  batch_size=BATCH, num_workers=NW),    "test":  DataLoader(Sen1Floods11(splits["test"]), batch_size=BATCH, num_workers=NW),}xb, yb, mb, _ = next(iter(loaders["train"]))print(f"batch  x{tuple(xb.shape)}  y{tuple(yb.shape)}  mask{tuple(mb.shape)}")print(f"water pixels in batch: {yb[mb>0].mean():.1%}   invalid: {(mb==0).float().mean():.1%}")

### 4b · Class imbalanceWater is a minority class — typically 10–25% of valid pixels. Left alone, the modelcan reach high pixel accuracy by predicting "dry" everywhere, which is exactly thefailure that matters (a missed flood, not a false one). Two counters below:- **`pos_weight`** in the BCE term, computed from the actual training distribution.- **A Dice term**, which is a set-overlap measure and therefore indifferent to the  size of the negative class.

In [ ]:
# Sample the training set to get the real positive rate.pos = tot = 0for i in range(min(len(splits["train"]), 60)):    _, y, m, _ = Sen1Floods11(splits["train"])[i]    pos += float(y[m > 0].sum()); tot += float((m > 0).sum())pos_rate = pos / max(tot, 1)POS_WEIGHT = float(np.clip((1 - pos_rate) / max(pos_rate, 1e-6), 1.0, 10.0))print(f"water = {pos_rate:.1%} of valid pixels  ->  pos_weight = {POS_WEIGHT:.2f}")print("(clipped at 10 — an unclipped weight on a rare class destabilises early training)")

## 5 · Loss — masked BCE + soft Dice

In [ ]:
def masked_loss(logits, target, valid, pos_weight: float, dice_w: float = 0.5):    """BCE + soft Dice, both computed only over observed pixels.    The mask is the load-bearing part. Sen1Floods11 marks unobserved pixels -1; if    they entered the loss as negatives, the model would learn "no data => dry",    which in production means a cloud-edge or layover shadow reads as safe ground.    """    w = torch.as_tensor(pos_weight, device=logits.device)    bce_map = F.binary_cross_entropy_with_logits(        logits, target, pos_weight=w, reduction="none")    bce = (bce_map * valid).sum() / valid.sum().clamp(min=1)    prob = torch.sigmoid(logits) * valid    tgt = target * valid    inter = (prob * tgt).sum(dim=(1, 2, 3))    union = prob.sum(dim=(1, 2, 3)) + tgt.sum(dim=(1, 2, 3))    dice = 1.0 - ((2 * inter + 1.0) / (union + 1.0)).mean()    return (1 - dice_w) * bce + dice_w * dice@torch.no_grad()def evaluate(model, loader, threshold: float = 0.5) -> dict:    """IoU, F1, precision, recall over valid pixels — pooled across the split."""    model.eval()    tp = fp = fn = 0.0    for x, y, m, _ in loader:        x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)        pred = (torch.sigmoid(model(x)) > threshold).float() * m        tgt = y * m        tp += float((pred * tgt).sum())        fp += float((pred * (1 - tgt) * m).sum())        fn += float(((1 - pred) * tgt * m).sum())    iou = tp / max(tp + fp + fn, 1)    prec = tp / max(tp + fp, 1)    rec = tp / max(tp + fn, 1)    f1 = 2 * prec * rec / max(prec + rec, 1e-9)    return {"iou": iou, "f1": f1, "precision": prec, "recall": rec}

## 6 · TrainRoughly 40 epochs. `AdamW` with cosine decay; gradient clipping because BatchNorm plusa small batch on a rare class can spike. Checkpoint selection is on **validation IoU**,not loss — IoU is what the product cares about.**Expected wall-clock.** Measured on an M4 Pro (MPS) at 512², batch 1:**~0.23 s per optimiser step**. With ~380 training chips at batch 8 that is ~48 stepsper epoch, so roughly **1–2 min/epoch → 40–80 min for 40 epochs**. A Colab T4 isbroadly comparable; CPU-only is several hours.If that is longer than you want, `EPOCHS = 20` usually gets most of the way — thecosine schedule and best-checkpoint selection mean stopping early is safe.

In [ ]:
import copy, math, timeEPOCHS = 40LR = 3e-4model = SARFloodUNet().to(DEVICE)opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)best = {"iou": -1.0, "state": None, "epoch": -1}history = []for epoch in range(1, EPOCHS + 1):    model.train()    running, seen, t0 = 0.0, 0, time.perf_counter()    for x, y, m, _ in loaders["train"]:        x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)        opt.zero_grad(set_to_none=True)        loss = masked_loss(model(x), y, m, POS_WEIGHT)        loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        opt.step()        running += float(loss) * x.size(0); seen += x.size(0)    sched.step()    val = evaluate(model, loaders["val"])    history.append({"epoch": epoch, "loss": running / max(seen, 1), **val})    if val["iou"] > best["iou"]:        # deepcopy on CPU so the checkpoint is device-independent        best = {"iou": val["iou"], "epoch": epoch,                "state": copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})}        marker = "  <-- best"    else:        marker = ""    print(f"epoch {epoch:3}/{EPOCHS}  loss {running/max(seen,1):.4f}  "          f"val IoU {val['iou']:.3f}  F1 {val['f1']:.3f}  "          f"P {val['precision']:.3f} R {val['recall']:.3f}  "          f"{time.perf_counter()-t0:.0f}s{marker}")print(f"\nbest val IoU {best['iou']:.3f} at epoch {best['epoch']}")model.load_state_dict(best["state"])model.to(DEVICE)

## 7 · Threshold selection`mean_fraction()` in the backend defaults to `0.5`. If a different operating point ismaterially better on validation, that is worth knowing — but **pick it on validation,never on test**, or the final number stops being an honest estimate.

In [ ]:
print("threshold   IoU     F1      precision  recall")best_t, best_t_iou = 0.5, -1.0for t in [0.3, 0.4, 0.5, 0.6, 0.7]:    m = evaluate(model, loaders["val"], threshold=t)    star = ""    if m["iou"] > best_t_iou:        best_t_iou, best_t, star = m["iou"], t, "  <--"    print(f"  {t:.2f}     {m['iou']:.3f}   {m['f1']:.3f}   "          f"{m['precision']:.3f}      {m['recall']:.3f}{star}")print(f"\nbest validation threshold: {best_t}")if abs(best_t - 0.5) > 1e-9:    print(f"NOTE: {best_t} beats the backend default of 0.5. To use it you must pass\n"          f"      threshold={best_t} in `mean_fraction()` — the weights alone will not\n"          f"      apply it. If you would rather not touch backend code, keep 0.5.")

## 8 · The ship/no-ship gate — trained model vs. the −16 dB heuristicThis is the decision cell. It evaluates both on **Nigeria**, which has been untoucheduntil now: no gradients, no hyper-parameter selection.Per §8.6 of the audit, a trained model ships only if it beats the physical threshold itreplaces. The backend already degrades to that threshold cleanly, so keeping theheuristic costs nothing — whereas shipping a worse model that reports `confidence 0.88`is actively harmful.

In [ ]:
GATE_SPLIT = "test" if splits["test"] else "val"if GATE_SPLIT == "val":    print("!! No Nigeria chips — falling back to the validation split.\n"          "   This is a weaker claim: those regions influenced early stopping.\n")@torch.no_grad()def evaluate_heuristic(loader, threshold_db: float = -16.0) -> dict:    """The -16 dB rule, scored identically so the comparison is apples-to-apples.    Note it is applied to RAW dB, not the standardised tensor — that is how    `indices.sar_water_mask` works in production.    """    tp = fp = fn = 0.0    for item in (splits[GATE_SPLIT]):        with rasterio.open(item["chip"]) as src:            vv = src.read(1).astype("float32")        with rasterio.open(item["label"]) as src:            lab = src.read(1).astype("float32")        vv = np.where(vv < -900, np.nan, vv)        valid = (lab != NO_DATA) & np.isfinite(vv)        pred = (vv < threshold_db)[valid]        tgt = (lab == 1)[valid]        tp += float((pred & tgt).sum()); fp += float((pred & ~tgt).sum()); fn += float((~pred & tgt).sum())    iou = tp / max(tp + fp + fn, 1); prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)    return {"iou": iou, "f1": 2*prec*rec/max(prec+rec, 1e-9), "precision": prec, "recall": rec}trained = evaluate(model, loaders[GATE_SPLIT], threshold=best_t)heuristic = evaluate_heuristic(loaders[GATE_SPLIT])print(f"Held-out {GATE_SPLIT.upper()} "      f"({sorted({p['region'] for p in splits[GATE_SPLIT]})}, "      f"{len(splits[GATE_SPLIT])} chips)\n")print(f"{'metric':11} {'-16 dB':>9} {'trained':>9} {'delta':>9}")for k in ("iou", "f1", "precision", "recall"):    d = trained[k] - heuristic[k]    print(f"{k:11} {heuristic[k]:9.3f} {trained[k]:9.3f} {d:+9.3f}")SHIP = trained["iou"] > heuristic["iou"]print("\n" + "="*62)if SHIP:    gain = (trained["iou"] - heuristic["iou"]) / max(heuristic["iou"], 1e-9)    print(f"PASS — trained model beats the heuristic on IoU by {gain:+.0%}.")    print("Export the weights in the next cell.")else:    print("FAIL — the trained model does NOT beat the -16 dB heuristic.")    print("Do NOT ship these weights. The backend's fallback is the better model,")    print("and confidence 0.88 over a worse predictor would be a real regression.")    print("\nWorth trying: more epochs, dice_w=0.3, LR 1e-4, or adding the")    print("WeaklyLabeled split (~4k chips) for pre-training.")print("="*62)

## 9 · Export `sar_flood.pt`Saved as a **plain state dict**, because `inference._load` calls`torch.load(..., weights_only=True)` — which refuses a pickled `nn.Module`. Tensors aremoved to CPU so the file loads on a CPU-only VPS.

In [ ]:
OUT = pathlib.Path("sar_flood.pt")if not SHIP:    print("Gate failed — writing to sar_flood_REJECTED.pt so it cannot be deployed by accident.")    OUT = pathlib.Path("sar_flood_REJECTED.pt")state = {k: v.detach().cpu() for k, v in model.state_dict().items()}torch.save(state, OUT)print(f"wrote {OUT}  ({OUT.stat().st_size/1e6:.2f} MB)")# --- verify it loads EXACTLY as the backend loads it ---------------------------probe = SARFloodUNet()loaded = torch.load(OUT, map_location="cpu", weights_only=True)   # same call as _load()probe.load_state_dict(loaded)                                     # strict=True by defaultprobe.eval()with torch.inference_mode():    out = probe(torch.randn(1, 2, 512, 512))assert out.shape == (1, 1, 512, 512), out.shapeprint(f"round-trip OK — output {tuple(out.shape)}, "      f"prob range [{torch.sigmoid(out).min():.3f}, {torch.sigmoid(out).max():.3f}]")# Parity check: reproduce predict_flood()'s preprocessing end to end.sample = splits[GATE_SPLIT][0] if splits[GATE_SPLIT] else splits["val"][0]with rasterio.open(sample["chip"]) as src:    raw = src.read().astype("float32")raw = np.where(raw < -900, np.nan, raw)vv_db, vh_db = raw[0], raw[1]stack = np.stack([_standardize(vv_db, 0), _standardize(vh_db, 1)], axis=0)with torch.inference_mode():    prob = torch.sigmoid(probe(torch.from_numpy(stack).unsqueeze(0))).squeeze().numpy()prob = np.where(np.isfinite(vv_db), prob, np.nan)          # backend re-masks like thisfrac = float(np.count_nonzero(prob[np.isfinite(prob)] > 0.5) / max(np.isfinite(prob).sum(), 1))print(f"serving-path parity on {sample['id']}: inundated_fraction = {frac:.3f}")

## 10 · Visual sanity checkNumbers can pass while the output is structurally wrong. Look at a few.

In [ ]:
import matplotlib.pyplot as pltshow = (splits[GATE_SPLIT] or splits["val"])[:4]fig, axes = plt.subplots(len(show), 4, figsize=(14, 3.4 * len(show)))axes = np.atleast_2d(axes)for row, item in enumerate(show):    with rasterio.open(item["chip"]) as src:        raw = np.where(src.read().astype("float32") < -900, np.nan, src.read().astype("float32"))    with rasterio.open(item["label"]) as src:        lab = src.read(1).astype("float32")    vv = raw[0]    stack = np.stack([_standardize(vv, 0), _standardize(raw[1])], axis=0)    with torch.inference_mode():        p = torch.sigmoid(probe(torch.from_numpy(stack).unsqueeze(0))).squeeze().numpy()    for col, (img, title, kw) in enumerate([        (vv, f"{item['id']}\nVV (dB)", dict(cmap="gray")),        (np.where(lab == NO_DATA, np.nan, lab), "label (grey = no data)", dict(cmap="Blues", vmin=0, vmax=1)),        (sar_water_mask(vv), "-16 dB heuristic", dict(cmap="Blues", vmin=0, vmax=1)),        (np.where(np.isfinite(vv), p, np.nan), "trained probability", dict(cmap="Blues", vmin=0, vmax=1)),    ]):        ax = axes[row, col]        ax.imshow(img, **kw); ax.set_title(title, fontsize=9); ax.axis("off")plt.tight_layout(); plt.show()

## 11 · Get the file onto your VPSThe artefact is **one ~1.9 MB file**. Nothing else transfers, and the runtime nevertrains or needs a GPU.### Where it goes```backend/app/ml/weights/sar_flood.pt````docker-compose.yml` already mounts that directory read-only into `api`, `worker` and`worker-analyst`:```yaml- ./backend/app/ml/weights:/app/app/ml/weights:ro```So the deployment step is: copy the file, restart. Nothing rebuilds.```bashscp sar_flood.pt user@vps:/srv/shelter/backend/app/ml/weights/ssh user@vps 'cd /srv/shelter && docker compose restart api worker worker-analyst'# Confirm it was picked up — "trained", not "heuristic-fallback":curl -s localhost:8000/api/v1/health | jq '.models'````.pt` files are gitignored (`.gitignore` line 20), so this never lands in the repo.Keep a copy somewhere durable — a release asset or object storage — because it is notreproducible from source alone.### What changes when it lands| | Before | After ||---|---|---|| Flood detection | `−16 dB` threshold | `SARFloodUNet` || `AnalystResult.confidence` | `0.55` | `0.88` || Max severity reachable | **`WATCH`** (0.55 < `CONFIDENCE_ESCALATION_FLOOR` 0.65) | up to `EMERGENCY` |That third row is the one to sit with. Today an untrained deployment **structurallycannot** raise an EMERGENCY. Installing these weights removes that interlock — so abug in them can now page people. That is precisely why cell 8 gates on held-outNigerian scenes rather than on validation loss.### Cell-level downloadColab:```pythonfrom google.colab import files; files.download("sar_flood.pt")```Kaggle: the file appears under `/kaggle/working` in the Output tab.Local Mac: it is in the notebook's working directory already.

## Appendix · If the gate failsIn order of expected value:1. **Pre-train on the weakly-labelled split.** Same bucket,   `v1.1/data/flood_events/WeaklyLabeled/` — ~4,300 chips labelled by an automated   Sentinel-2 water index. Noisier, but ~10× the data. Pre-train there for ~20 epochs,   then fine-tune on hand labels. This is the single biggest lever.2. **Loosen the Dice weight** to `dice_w=0.3`, and try `LR=1e-4` with `EPOCHS=60`.3. **Add permanent water as a third input channel** (JRC GSW, §6 step 1). This helps   for the same reason it helps in production — but it changes `in_channels` to 3, so   the backend's `_standardize` stack and `SARFloodUNet(in_channels=2)` must change   together. **A three-channel checkpoint will not load into the current backend.**4. **Accept the heuristic.** A genuinely acceptable outcome. 446 chips across 11   regions is a small dataset, and if the model cannot beat calibrated physics on   held-out Nigerian terrain, the physics is the better product. `confidence 0.55` and   the `WATCH` ceiling then remain correct rather than pessimistic.